# MetroDemand Analytics | Registro del modelo definitivo
El servicio recibe variables de calendario ya calculadas; no infiere festivos ni horarios de operación. Ejecutar en orden. No reentrena ni modifica las tablas fuente.

## 0. Configuración

In [ ]:
import mlflow
import pandas as pd
import numpy as np
import tempfile
from pathlib import Path
from mlflow.models import infer_signature

RUN = spark.table("workspace.analitica.metrodemanda_predicciones_ago_dic2026").select("run_id_modelo").distinct().collect()
assert len(RUN) == 1
RUN = RUN[0][0]
REGISTRO = "workspace.analitica.metrodemanda_modelo_definitivo"
EXPERIMENTO = f"/Users/{spark.sql('SELECT current_user() AS usuario').first().usuario}/MetroDemand_Analytics"
mlflow.set_experiment(EXPERIMENTO)
mlflow.set_registry_uri("databricks-uc")
run = mlflow.get_run(RUN)
assert run.data.params['algoritmo'] == 'Mediana histórica'
GLOBAL = float(run.data.params['mediana_global'])
NIVELES = [['linea','hora_num','dia_semana_num','es_festivo'],['linea','hora_num','dia_semana_num'],['linea','hora_num'],['linea']]
ENTRADAS = NIVELES[0] + ['en_horario_operacion']


## 1. Recuperación y empaquetado

In [ ]:
tablas = [spark.table(f"workspace.analitica.metrodemanda_mediana_{RUN}_nivel_{i}").toPandas() for i in range(4)]
for i, tabla in enumerate(tablas):
    assert not tabla.duplicated(NIVELES[i]).any()
    assert tabla[f'mediana_{i}'].notna().all()

class MetroDemand(mlflow.pyfunc.PythonModel):
    def __init__(self, tablas, niveles, global_):
        self.niveles = niveles
        self.global_ = global_
        self.mapas = [dict(zip(t[k].itertuples(index=False, name=None), t[f'mediana_{i}'])) for i,(t,k) in enumerate(zip(tablas,niveles))]
    def predict(self, context, model_input, params=None):
        resultados = []
        for r in model_input.to_dict('records'):
            if r['en_horario_operacion'] not in (0,1) or r['es_festivo'] not in (0,1) or not 4 <= r['hora_num'] <= 23 or not 1 <= r['dia_semana_num'] <= 7:
                raise ValueError('Variables de calendario fuera de rango')
            v = 0.0
            if r['en_horario_operacion'] == 1:
                v = self.global_
                for k,m in zip(self.niveles,self.mapas):
                    clave = tuple(r[c] for c in k)
                    if clave in m:
                        v = m[clave]
                        break
            resultados.append(int(np.floor(v + 0.5)))
        return pd.DataFrame({'demanda_predicha': np.asarray(resultados,dtype='int64')})

modelo = MetroDemand(tablas,NIVELES,GLOBAL)


## 2. Validación contra el pronóstico original

In [ ]:
referencia = spark.table('workspace.analitica.metrodemanda_predicciones_ago_dic2026').filter(f"run_id_modelo = '{RUN}'").select(*ENTRADAS,'demanda_predicha').toPandas()
assert len(referencia) == 15300, 'El pronóstico no corresponde al run seleccionado'
X = referencia[ENTRADAS].copy()
for c in ENTRADAS[1:]:
    X[c] = X[c].astype('int64')
y = modelo.predict(None,X)
assert np.array_equal(y.demanda_predicha.to_numpy(),referencia.demanda_predicha.to_numpy()), 'Predicciones diferentes: detener registro'
print('Equivalencia exacta: 15.300 predicciones')


## 3. Registro y recuperación MLflow

In [ ]:
with mlflow.start_run(run_name='Empaquetado modelo definitivo'):
    mlflow.log_param('run_entrenamiento',RUN)
    mlflow.log_metric('registros_equivalentes',len(X))
    info = mlflow.pyfunc.log_model(artifact_path='modelo',python_model=modelo,signature=infer_signature(X,y),input_example=X.head(5))
    recuperado = mlflow.pyfunc.load_model(info.model_uri)
    assert np.array_equal(recuperado.predict(X).demanda_predicha.to_numpy(),y.demanda_predicha.to_numpy())
    version = mlflow.register_model(info.model_uri,REGISTRO)
print('Modelo:',REGISTRO,'Versión:',version.version)


## 4. Prueba de consumo del modelo registrado

In [ ]:
registrado = mlflow.pyfunc.load_model(f'models:/{REGISTRO}/{version.version}')
assert np.array_equal(registrado.predict(X).demanda_predicha.to_numpy(),y.demanda_predicha.to_numpy())
print('Modelo registrado y validado')
display(pd.concat([X.head(10),registrado.predict(X.head(10))],axis=1))
print('Ejemplo de entrada REST:')
print(X.head(2).to_json(orient='split',index=False))
